In [50]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
import warnings
from features import get_summarized_season
from clustering import group_pca
from sklearn.cluster import KMeans
# from utils import get_mens_regular_season, get_mens_tourney, get_mens_seeeds, get_womens_regular_season, get_womens_tourney, get_womens_seeeds

warnings.filterwarnings("ignore")

data_dir = "/Users/loganheydt/Desktop/Data/GitHub/Kaggle_competitions/March_Machine_Learning_Mania_2026/data"


In [51]:
M_regular_results = pd.read_csv(f"{data_dir}/MRegularSeasonDetailedResults.csv")
M_tourney_results = pd.read_csv(f"{data_dir}/MNCAATourneyDetailedResults.csv")
M_seeds = pd.read_csv(f"{data_dir}/MNCAATourneySeeds.csv")

W_regular_results = pd.read_csv(f"{data_dir}/WRegularSeasonDetailedResults.csv")
W_tourney_results = pd.read_csv(f"{data_dir}/WNCAATourneyDetailedResults.csv")
W_seeds = pd.read_csv(f"{data_dir}/WNCAATourneySeeds.csv")

In [52]:
# using the last 20 years of data, change it needed
# cut_off = 2005
cut_off = 1600

regular_results = M_regular_results.loc[M_regular_results['Season'] >= cut_off]
tourney_results = M_tourney_results.loc[M_tourney_results['Season'] >= cut_off]
seeds = M_seeds.loc[M_seeds['Season'] >= cut_off]

# regular_results = W_regular_results.loc[W_regular_results['Season'] >= cut_off]
# tourney_results = W_tourney_results.loc[W_tourney_results['Season'] >= cut_off]
# seeds = W_seeds.loc[W_seeds['Season'] >= cut_off]

In [ ]:
regular_results.head()

In [ ]:
tourney_results.head()

In [55]:
max_day = tourney_results['DayNum'].max()

In [ ]:
seeds.head()

In [ ]:
seeds["seed"] = seeds["Seed"].apply(lambda x: int(x[1:3]))
seeds["division"] = seeds["Seed"].apply(lambda x: x[0])

seeds.head()

In [58]:
champs = tourney_results.loc[tourney_results['DayNum']==max_day]
champs = champs.rename(columns={'WTeamID': 'TeamID'})
champs = champs.merge(seeds, how='left', on=['Season', 'TeamID'])

In [ ]:
champs['seed'].value_counts().sort_index()

In [ ]:
seed_pct = (
    champs['seed']
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

plt.figure()
plt.bar(seed_pct.index.astype(str), seed_pct.values)

plt.xlabel("Seed")
plt.ylabel("Percentage (%)")
plt.title("Champion Seed Distribution (%)")
plt.xticks(rotation=45)

plt.show()

In [ ]:
div_pct = (
    champs['division']
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

plt.figure()
plt.bar(div_pct.index.astype(str), div_pct.values)

plt.xlabel("Division")
plt.ylabel("Percentage (%)")
plt.title("Champion Division Distribution (%)")
plt.xticks(rotation=45)

plt.show()

In [62]:
tourney_results = tourney_results.merge(
    seeds[['Season', 'TeamID', 'seed']],
    left_on=['Season', 'WTeamID'],
    right_on=['Season', 'TeamID'],
    how='left'
).rename(columns={'seed': 'Wseed'})

tourney_results = tourney_results.merge(
    seeds[['Season', 'TeamID', 'seed']],
    left_on=['Season', 'LTeamID'],
    right_on=['Season', 'TeamID'],
    how='left'
).rename(columns={'seed': 'Lseed'})

In [63]:
tourney_results['seed_dif'] = tourney_results['Wseed'] - tourney_results['Lseed']
tourney_results['point_dif'] = tourney_results['WScore'] - tourney_results['LScore']

In [ ]:
tourney_results[['seed_dif', 'point_dif']].corr()

In [65]:
mens_season_stats = get_summarized_season(df=M_regular_results, seeds=M_seeds)

In [ ]:
mens_season_stats.columns

## PCA

In [67]:
reduced_df, pca_group_dict, fitted_objects = group_pca(
    df=mens_season_stats
)

In [68]:
# Elbow method
inertias = []
mapping1 = {}
K = range(1,10)

for k in K:
    model = KMeans(k, random_state=7).fit(reduced_df.values)
    labels = model.labels_
    centers = model.cluster_centers_

    inertias.append(model.inertia_)

    mapping1[k] = inertias[-1]

In [ ]:
print("Inertia values:")
for key, val in mapping1.items():
    print(f'{key} : {val}')

plt.plot(K, inertias, 'bx-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('The Elbow Method')
plt.show()

In [ ]:
out = add_team_clusters(mens_season_stats)

In [ ]:
out

In [47]:
# NEXT STEPS:


# RE-RUN CLUSTERING EXPLORATION

In [48]:
# 1. Build season stats -  DONE
# 2. Create efficiency metrics - DONE
# 3. Run PCA - DONE
# 4. Cluster teams - DONE
# 5. Build matchup dataset
# 6. Train XGBoost
# 7. Calibrate probabilities

# Ideas
### Compare avg regular season stats (scoring, defense, differentials)
### Understand how each team played against other types of team
###    - Did they play well against good defenses? 
###    - Are they more prone to lose if the other team is known for lots of offensive boards?